# Week 1 · Day 1 — Riassunto della documentazione Aviatrix

**Corso:** AI Engineer Core Track (LLM Engineering) — esercizio personale basato sul lab Day 1 di Ed Donner.

**Obiettivo:** leggere la home page di [docs.aviatrix.com](https://docs.aviatrix.com/) e farne un riassunto chiaro con GPT, come se lo spiegassi a un collega che non conosce il prodotto.

**Lo schema è lo stesso del lab:**

```
URL → scraper (testo del sito) → messages (system + user) → GPT → riassunto in Markdown
```

Cosa cambia rispetto al lab di Ed:
- lo **scraper** è scritto dentro il notebook, così si vede ogni passo;
- il **system prompt** è da analista tecnico, non sarcastico;
- c'è un **bonus** che usa `llms.txt`, un indice della documentazione pensato apposta per gli LLM.

Esegui le celle **dall'alto verso il basso** con Shift+Invio.

## 1. Import

- `requests` scarica le pagine web
- `BeautifulSoup` estrae il testo dall'HTML
- `dotenv` + `os` leggono la chiave API dal file `.env`
- `OpenAI` è il client per chiamare GPT
- `Markdown` + `display` mostrano la risposta formattata

In [ ]:
import os
import requests
from bs4 import BeautifulSoup
from dotenv import load_dotenv
from openai import OpenAI
from IPython.display import Markdown, display

## 2. Chiave API

La chiave sta nel file `.env` nella cartella principale del progetto, con una riga così:

```
OPENAI_API_KEY=sk-proj-...
```

Il `.env` deve essere nel `.gitignore`: la chiave non va mai su GitHub.

In [ ]:
load_dotenv(override=True)
api_key = os.getenv("OPENAI_API_KEY")

if not api_key:
    print("Nessuna chiave trovata: controlla il file .env")
elif not api_key.startswith("sk-proj-"):
    print("Chiave trovata, ma non inizia con sk-proj-: verifica che sia quella giusta")
elif api_key.strip() != api_key:
    print("La chiave ha spazi all'inizio o alla fine: toglili dal file .env")
else:
    print("Chiave API trovata!")

openai = OpenAI()
MODEL = "gpt-4.1-mini"

## 3. Lo scraper

È la stessa logica di `scraper.py` di Ed, scritta qui per vederla da vicino:

1. `requests.get` scarica la pagina; l'header `User-Agent` fa sembrare la richiesta quella di un browser.
2. `BeautifulSoup` legge l'HTML.
3. Togliamo le parti inutili: script, stili, immagini, campi input, e in più **menu di navigazione, header e footer**, che in un sito di documentazione sono tantissimi.
4. Teniamo solo il testo e lo tagliamo a `max_chars` caratteri.

Ho alzato il limite a 5.000 caratteri (Ed usa 2.000) perché le pagine di documentazione sono dense.

In [ ]:
HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
                  "(KHTML, like Gecko) Chrome/117.0.0.0 Safari/537.36"
}


def fetch_website_contents(url, max_chars=5_000):
    """Restituisce titolo + testo leggibile della pagina, tagliato a max_chars."""
    response = requests.get(url, headers=HEADERS, timeout=30)
    response.raise_for_status()  # errore chiaro se il sito risponde 403, 404...
    soup = BeautifulSoup(response.content, "html.parser")

    title = soup.title.string.strip() if soup.title and soup.title.string else "Nessun titolo"

    if soup.body:
        for irrelevant in soup.body(["script", "style", "img", "input", "nav", "header", "footer", "svg"]):
            irrelevant.decompose()
        text = soup.body.get_text(separator="\n", strip=True)
    else:
        text = ""

    return (title + "\n\n" + text)[:max_chars]

## 4. Guardiamo cosa vede lo scraper

Prima di chiamare GPT conviene **guardare il testo grezzo**. Se qui c'è poco o niente, il problema è lo scraper, non il modello.

In [ ]:
URL = "https://docs.aviatrix.com/"

aviatrix = fetch_website_contents(URL)
print(f"Caratteri estratti: {len(aviatrix)}\n")
print(aviatrix)

## 5. I prompt

- **System prompt**: chi è il modello e come deve rispondere.
- **User prompt**: il materiale da riassumere.

Qui chiedo una struttura fissa, così il riassunto è sempre ordinato e confrontabile tra pagine diverse.

In [ ]:
system_prompt = """
Sei un analista tecnico esperto di cloud networking.
Ricevi il testo di una pagina di documentazione e la spieghi a un collega
che non conosce il prodotto.
Ignora menu, link di navigazione e testo ripetuto.
Rispondi in italiano, in Markdown, senza blocchi di codice, con queste sezioni:

## Cos'è
Due o tre frasi: che prodotto è e quale problema risolve.

## Cosa trovi nella documentazione
Elenco puntato delle aree principali, una riga ciascuna.

## Da dove iniziare
Il percorso consigliato per un principiante.

## Novità
Eventuali aggiornamenti o release citati; se non ce ne sono, scrivi "Nessuna novità indicata".
"""

user_prompt_prefix = """
Ecco il contenuto della home page di un sito di documentazione.
Riassumilo seguendo la struttura richiesta.

"""

## 6. I messaggi

La lista nel formato che OpenAI si aspetta. Eseguendo la cella vedi esattamente cosa riceverà GPT.

In [ ]:
def messages_for(website):
    return [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt_prefix + website},
    ]

messages_for(aviatrix)

## 7. Chiamata a GPT e visualizzazione

In [ ]:
def summarize(url):
    website = fetch_website_contents(url)
    response = openai.chat.completions.create(
        model=MODEL,
        messages=messages_for(website),
    )
    return response.choices[0].message.content


def display_summary(url):
    display(Markdown(summarize(url)))

In [ ]:
display_summary(URL)

## 8. Proviamo altre pagine della documentazione

La stessa funzione funziona su qualsiasi pagina del sito. Copia un indirizzo dalla documentazione e incollalo qui.

In [ ]:
# Sostituisci con una pagina che ti interessa della documentazione Aviatrix
altra_pagina = "https://docs.aviatrix.com/"

display_summary(altra_pagina)

## 9. Bonus: `llms.txt`

Molti siti di documentazione moderni pubblicano un file `llms.txt`: un indice in Markdown di tutte le pagine, con titolo, link e una breve descrizione. È pensato proprio per essere letto dagli LLM.

Per Aviatrix si trova in [docs.aviatrix.com/llms.txt](https://docs.aviatrix.com/llms.txt). Invece di riassumere una sola pagina, possiamo far riassumere a GPT **la mappa dell'intera documentazione**.

Il file è già testo, quindi non serve BeautifulSoup: basta `requests`.

In [ ]:
def fetch_text(url, max_chars=15_000):
    """Scarica un file di testo (come llms.txt) e lo taglia a max_chars."""
    response = requests.get(url, headers=HEADERS, timeout=30)
    response.raise_for_status()
    return response.text[:max_chars]


llms_txt = fetch_text("https://docs.aviatrix.com/llms.txt")
print(f"Caratteri: {len(llms_txt)}\n")
print(llms_txt[:1_500])  # mostriamo solo l'inizio

In [ ]:
map_system_prompt = """
Sei un tutor che aiuta un principiante a orientarsi in una documentazione tecnica.
Ricevi l'indice completo del sito (formato llms.txt: titolo, link, descrizione).
Rispondi in italiano, in Markdown, senza blocchi di codice:
1. In 3 frasi, di cosa parla la documentazione nel suo insieme.
2. Le macro-aree in cui è organizzata, con una riga ciascuna.
3. Un percorso di lettura di 5 pagine per chi parte da zero, con i link.
"""

response = openai.chat.completions.create(
    model=MODEL,
    messages=[
        {"role": "system", "content": map_system_prompt},
        {"role": "user", "content": "Ecco l'indice della documentazione:\n\n" + llms_txt},
    ],
)
display(Markdown(response.choices[0].message.content))

## Cosa ho imparato

- Lo schema del lab di Ed (**URL → scraper → messages → GPT → Markdown**) funziona su qualsiasi sito leggibile senza JavaScript.
- **Guardare il testo grezzo** prima di chiamare GPT fa capire subito se il problema è lo scraper o il prompt.
- Togliere `nav`, `header` e `footer` è importante nei siti di documentazione, dove i menu occupano gran parte del testo.
- Un **system prompt con struttura fissa** rende i riassunti ordinati e confrontabili.
- `llms.txt` è una scorciatoia preziosa: un indice già pronto per gli LLM.

**Limiti:** il testo viene tagliato (5.000 caratteri per le pagine, 15.000 per `llms.txt`), quindi il riassunto copre solo la parte iniziale. Siti costruiti interamente in JavaScript richiederebbero Selenium o Playwright.

**Possibili estensioni:** riassumere automaticamente le prime N pagine di `llms.txt`; confrontare la documentazione di due prodotti; provare un modello gratuito con Ollama.